# Uniformity of the survey as months accumulate (DP2 / pre-LSST visits, WFD without DDF)

- creation date : 2026-10-09 (adapted to the DP2 / pre-LSST visits on 2026-10-31)
- input visits : `prelsst_20260607_visits.db` (DP2 / pre-LSST, April 2025 - July 2026)
- based on : `03_N_Per_Season.ipynb` (same WFD footprint, same SQL constraint, same custom-metric approach)

## Overview and outputs

Notebook `03_N_Per_Season` shows the cadence **season by season**. Static cosmology (large-scale structure, weak lensing, 3x2pt) instead needs a survey that is **uniform in number of visits / depth as the data accumulate**. There is no ready-made MAF metric for this, so this notebook builds the cumulative maps and quantifies their uniformity.

In the 10-year version the epochs were years. Here the DP2 / pre-LSST visits cover only about 15 months, so the epochs are **months**: 'after N months' means all the visits taken before `START_MJD + N months` (`START_MJD` = first day of the visits, 1 month = 365.25 / 12 days), for N = 1 ... `N_MONTHS` (computed from the database, about 16). One set of figures is produced **for each month**.

The season-based definition of the 10-year version (visits in the first N observing seasons of each sky position) is dropped: over this period only the seasons -1 to 1 can exist (see notebook 03), so it carries no information at a monthly resolution.

Steps:
1. Define a custom MAF metric (`CumulativeVisitsMetric`) returning, for each sky position, the cumulative number of visits and the cumulative coadded depth in one band, for N = 1 ... `N_MONTHS`.
2. Run it on the WFD pixels (no DDF) of the DP2 / pre-LSST visits.
3. Map the cumulative number of visits for each month (absolute and normalised to its median): one figure per month.
4. Quantify the uniformity versus the number of months (relative RMS, robust sigma, 5-95 percentile spread, fraction of pixels within 10% of the median).
5. Same for the coadded depth in one band.

**Note**: the WFD footprint comes from the scheduler goal map (`CurrentAreaMap`); the first months of the survey do not necessarily cover it, so many pixels can have no visit yet (`frac_zero`).

**Input**: the DP2 / pre-LSST visit database `prelsst_20260607_visits.db` (path set in `opsim_fname`).

**Outputs**: arrays and tables go to `DATA06_DDF_RollingUniformity/`; figures (PNG and PDF) go to `FIG06_DDF_RollingUniformity/`.

## 0. Imports

In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import healpy as hp
from IPython.display import display
from astropy.time import Time

import rubin_sim.maf as maf
from rubin_sim.maf import BaseMetric

from rubin_scheduler.scheduler.utils import CurrentAreaMap

## 1. Configuration

- `opsim_fname`: the DP2 / pre-LSST visit database.
- `START_MJD`, `N_MONTHS`: origin of the epochs (first day of the visits) and number of monthly epochs, both computed from the database (expected: April 2025 to July 2026).
- `MIN_EXP_TIME`: visits shorter than this are dropped (near-sun twilight visits), as in notebook 03.
- `DEPTH_FILTER`: band used for the coadded depth.
- `REL_MIN`, `REL_MAX`: colour range of the maps normalised to their median; `DM5_RANGE`: half colour range of the depth maps (mag).
- Output directories `DATA_DIR` / `FIG_DIR` are created.
- `epoch_end_label(n)`: calendar date of the end of month N, used in the titles.

In [ ]:
# DP2 / pre-LSST visit database
opsim_fname = "/Users/dagoret/DATA/OpSim/dp2/prelsst_20260607_visits.db"
# Same file as: os.path.join(os.environ["RUBIN_SIM_DATA_DIR"], "dp2", "prelsst_20260607_visits.db")
runName = os.path.split(opsim_fname)[-1].replace(".db", "")
print(opsim_fname)
print(runName)

NSIDE = 64  # HEALPix resolution
YEAR = 365.25  # days
MONTH = YEAR / 12  # days
MIN_EXP_TIME = 16  # s
DEPTH_FILTER = "i"  # band for the coadded depth
REL_MIN, REL_MAX = 0.5, 1.5
DM5_RANGE = 0.3  # mag

# Time span of the visits (expected: April 2025 to July 2026): origin and number of the monthly epochs
with sqlite3.connect(opsim_fname) as con:
    mjd_min, mjd_max, n_visits = con.execute(
        "select min(observationStartMJD), max(observationStartMJD), count(*) from observations"
    ).fetchone()
START_MJD = float(np.floor(mjd_min))
N_MONTHS = int(np.ceil((mjd_max - START_MJD) / MONTH))
print(f"Number of visits: {n_visits}")
print(f"First visit: {Time(mjd_min, format='mjd').iso}")
print(f"Last visit:  {Time(mjd_max, format='mjd').iso}")
print(f"START_MJD = {START_MJD}, N_MONTHS = {N_MONTHS}")

# Output directories: arrays/tables go to DATA_DIR, all figures go to FIG_DIR
DATA_DIR = "DATA06_DDF_RollingUniformity"
FIG_DIR = "FIG06_DDF_RollingUniformity"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)


def epoch_end_label(n):
    # Calendar date (YYYY-MM-DD) of the end of month N
    return Time(START_MJD + n * MONTH, format="mjd").iso[:10]


KIND_LABEL = {"n_time": "time-based", "m5_time": "time-based"}
WHEN = {"n_time": "visits before {date}", "m5_time": "depth before {date}"}

## 2. Custom MAF metric

`CumulativeVisitsMetric` works on one sky position. It keeps the visits longer than `min_exp_time` and returns a dictionary of arrays of length `n_months`:

- `n_time[k-1]`: number of visits taken before `mjd_start + k months`.
- `m5_time[k-1]`: coadded 5-sigma depth in `depth_filter` for the same visits, `1.25 log10(sum 10^(0.8 m5))`; NaN if there is no visit in that band yet.

A position that has no visit at all keeps masked in MAF; it is treated as 0 visits (and no depth) in the next sections.

In [ ]:
# allow re-running the cell: remove a previous registration of the metric
if "__main__.CumulativeVisitsMetric" in BaseMetric.registry:
    del BaseMetric.registry["__main__.CumulativeVisitsMetric"]


class CumulativeVisitsMetric(BaseMetric):
    """Cumulative number of visits and coadded depth after N months.

    Parameters
    ----------
    n_months : `int`
        Number of cumulative monthly epochs (1 ... n_months).
    depth_filter : `str`
        Band used for the coadded depth.
    min_exp_time : `float`
        Minimum exposure time (s) to count a visit (drops twilight visits).
    mjd_start : `float` or None
        Origin of the epochs (mjd). Default None uses START_MJD.
    """

    def __init__(
        self,
        n_months=12,
        depth_filter="i",
        min_exp_time=16,
        mjd_start=None,
        mjd_col="observationStartMJD",
        exp_time_col="visitExposureTime",
        m5_col="fiveSigmaDepth",
        filter_col="filter",
        metric_name="CumulativeVisits",
        **kwargs,
    ):
        if mjd_start is None:
            mjd_start = START_MJD
        self.mjd_start = mjd_start
        self.n_months = n_months
        self.depth_filter = depth_filter
        self.min_exp_time = min_exp_time
        self.mjd_col = mjd_col
        self.exp_time_col = exp_time_col
        self.m5_col = m5_col
        self.filter_col = filter_col
        super().__init__(
            col=[mjd_col, exp_time_col, m5_col, filter_col],
            units="#",
            metric_name=metric_name,
            metric_dtype="object",
            **kwargs,
        )

    @staticmethod
    def _coadd(m5):
        if len(m5) == 0:
            return np.nan
        return 1.25 * np.log10(np.sum(10.0 ** (0.8 * m5)))

    def run(self, data_slice, slice_point):
        n = self.n_months
        out = {
            "n_time": np.zeros(n),
            "m5_time": np.full(n, np.nan),
        }
        data = data_slice[data_slice[self.exp_time_col] > self.min_exp_time]
        if len(data) == 0:
            return out
        mjd = data[self.mjd_col]
        m5 = data[self.m5_col]
        # the filter column can be bytes or str depending on the reader: force str
        filt = np.asarray(data[self.filter_col]).astype(str)
        in_band = filt == self.depth_filter
        for k in range(1, n + 1):
            sel_t = mjd < self.mjd_start + k * MONTH
            out["n_time"][k - 1] = np.count_nonzero(sel_t)
            out["m5_time"][k - 1] = self._coadd(m5[sel_t & in_band])
        return out

## 3. Metric set-up on the WFD footprint (no DDF)

Same selection as notebook 03:
- WFD pixels from the scheduler labels (`lowdust`, `euclid_overlap`, `LMC_SMC`, `virgo`), used as a `HealpixSubsetSlicer`.
- The SQL constraint removes DDF, ToO and twilight visits, so only regular survey visits are counted. `coalesce` keeps the visits whose `scheduler_note` is empty (NULL).

A single constraint is used (all epochs are computed inside the metric), so the database is read only once. The cell also prints the values really stored in the band column, to check them against `DEPTH_FILTER`.

In [ ]:
skymap = CurrentAreaMap(nside=NSIDE)
maps, labels = skymap.return_maps()

wfdhpid = np.where(
    (labels == "lowdust") | (labels == "euclid_overlap") | (labels == "LMC_SMC") | (labels == "virgo")
)[0]
wfd_slicer = maf.HealpixSubsetSlicer(nside=NSIDE, hpid=wfdhpid)

constraint = (
    "coalesce(scheduler_note, '') not like '%dd%' and coalesce(scheduler_note, '') not like '%too%' "
    "and coalesce(scheduler_note, '') not like '%twi%'"
)
label = "WFD (no DDF)"

# Name of the band column: recent OpSim databases use 'band', older ones 'filter'.
# Print the values really stored in the database to check them against DEPTH_FILTER.
with sqlite3.connect(opsim_fname) as con:
    db_columns = [r[1] for r in con.execute("PRAGMA table_info(observations)")]
    BAND_COL = "band" if "band" in db_columns else "filter"
    for c in ("band", "filter"):
        if c in db_columns:
            counts = con.execute(f"SELECT [{c}], COUNT(*) FROM observations GROUP BY [{c}]").fetchall()
            print(f"column {c}: {dict(counts)}")
print("band column used by the metric:", BAND_COL)

metric = CumulativeVisitsMetric(
    n_months=N_MONTHS,
    depth_filter=DEPTH_FILTER,
    min_exp_time=MIN_EXP_TIME,
    mjd_start=START_MJD,
    filter_col=BAND_COL,
)
bundle = maf.MetricBundle(metric, wfd_slicer, constraint, info_label=label, run_name=runName)
g = maf.MetricBundleGroup({"cum": bundle}, opsim_fname, out_dir=DATA_DIR, results_db=None, verbose=True)

## 4. Run the metric and extract the cumulative maps

After running, the object array `bundle.metric_values` is unpacked into two arrays of shape `(N_MONTHS, n_pixels)` stored in the dictionary `cum` (`n_time`: number of visits, `m5_time`: coadded depth). `HPID` gives the HEALPix index of each WFD pixel. Pixels with no visit get 0 visits and NaN depth. The arrays are saved in `DATA_DIR`.

In [ ]:
g.run_all()

In [ ]:
HPID = np.asarray(wfd_slicer.slice_points["sid"])
npix = len(HPID)

mv = bundle.metric_values
mask = np.ma.getmaskarray(mv)

cum = {
    "n_time": np.zeros((N_MONTHS, npix)),
    "m5_time": np.full((N_MONTHS, npix), np.nan),
}
n_filled = 0
for ip in range(npix):
    if mask[ip]:
        continue
    v = mv.data[ip]
    if not isinstance(v, dict):
        continue
    n_filled += 1
    for key in cum:
        cum[key][:, ip] = v[key]

# The HealpixSubsetSlicer keeps the full-sky pixel list (pixels outside the WFD are masked):
# keep only the WFD pixels, otherwise most of the pixels are empty and the median is 0
keep = np.isin(HPID, wfdhpid)
HPID = HPID[keep]
for key in cum:
    cum[key] = cum[key][:, keep]
npix = len(HPID)
n_obs = np.count_nonzero(cum["n_time"][-1] > 0)
print(f"slicer pixels: {len(mask)}, WFD pixels kept: {npix}, with at least one visit: {n_obs}")
np.savez(os.path.join(DATA_DIR, f"cumulative_maps_{runName}.npz"), hpid=HPID, **cum)
if not np.isfinite(cum["m5_time"]).any():
    print(
        "WARNING: no finite depth, compare BAND_COL / DEPTH_FILTER with the values printed in the set-up cell"
    )

## 5. Uniformity statistics versus the number of months

For each month and over all WFD pixels (pixels with no visit counted as 0):

- `rms/mean`: standard deviation divided by the mean of the number of visits.
- `robust_sigma/median`: `1.4826 * MAD / median`, insensitive to the tails.
- `(p95-p5)/median`: spread between the 5th and 95th percentiles.
- `frac_within_10pc`: fraction of pixels within +/-10% of the median.
- `frac_zero`: fraction of pixels without any visit.

These indicators are NaN when the median is 0, which is expected in the first months if most of the footprint is not observed yet.

For the depth (`DEPTH_FILTER` band): standard deviation, robust sigma and 5-95 percentile spread in magnitudes, and the fraction of pixels within 0.05 mag of the median. Tables are saved as CSV in `DATA_DIR`.

In [ ]:
def stats_counts(x):
    med = np.median(x)
    mean = np.mean(x)
    p5, p95 = np.percentile(x, [5, 95])
    robust = 1.4826 * np.median(np.abs(x - med))
    ok = med > 0
    return {
        "median": med,
        "mean": mean,
        "rms/mean": np.std(x) / mean if mean > 0 else np.nan,
        "robust_sigma/median": robust / med if ok else np.nan,
        "(p95-p5)/median": (p95 - p5) / med if ok else np.nan,
        "frac_within_10pc": np.mean(np.abs(x / med - 1) < 0.10) if ok else np.nan,
        "frac_zero": np.mean(x == 0),
    }


def stats_depth(m5):
    ok = np.isfinite(m5)
    if ok.sum() == 0:
        return {
            "median_m5": np.nan,
            "std": np.nan,
            "robust_sigma": np.nan,
            "p95-p5": np.nan,
            "frac_within_0.05mag": np.nan,
            "frac_with_data": 0.0,
        }
    y = m5[ok]
    med = np.median(y)
    p5, p95 = np.percentile(y, [5, 95])
    return {
        "median_m5": med,
        "std": np.std(y),
        "robust_sigma": 1.4826 * np.median(np.abs(y - med)),
        "p95-p5": p95 - p5,
        "frac_within_0.05mag": np.mean(np.abs(y - med) < 0.05),
        "frac_with_data": ok.mean(),
    }


rows = []
for k in range(N_MONTHS):
    r = stats_counts(cum["n_time"][k])
    r.update(kind="n_time", n=k + 1, until=epoch_end_label(k + 1))
    rows.append(r)
df_counts = pd.DataFrame(rows)

rows = []
for k in range(N_MONTHS):
    r = stats_depth(cum["m5_time"][k])
    r.update(kind="m5_time", n=k + 1, until=epoch_end_label(k + 1))
    rows.append(r)
df_depth = pd.DataFrame(rows)

df_counts.to_csv(os.path.join(DATA_DIR, f"uniformity_nvisits_{runName}.csv"), index=False)
df_depth.to_csv(os.path.join(DATA_DIR, f"uniformity_depth_{DEPTH_FILTER}_{runName}.csv"), index=False)

display(df_counts.round(3))
display(df_depth.round(3))

## 6. Sky maps of the cumulative number of visits, one figure per month

`to_fullsky` puts the WFD pixel values on a full HEALPix map (other pixels are `hp.UNSEEN`). For each month, `plot_month` draws two Mollweide maps and saves them (PNG + PDF) in `FIG_DIR`:

- left: absolute cumulative number of visits, same colour scale for all months (0 to the 99.5th percentile of the last month), so that the accumulation is visible;
- right: the same map divided by its median (colour range `REL_MIN` - `REL_MAX`), which shows the non-uniformity at that epoch independently of the total number of visits. The median is replaced by 1 when it is 0, so the right panel then shows the number of visits itself.

In [ ]:
CMAP_DIV = plt.get_cmap("RdBu_r")


def to_fullsky(values):
    m = np.full(hp.nside2npix(NSIDE), hp.UNSEEN)
    good = np.isfinite(values)
    m[HPID[good]] = values[good]
    return m


def save_fig(fig, name):
    base = os.path.join(FIG_DIR, name)
    fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
    fig.savefig(base + ".pdf", bbox_inches="tight")


def plot_month(k, vmax_abs):
    n = k + 1
    kind = "n_time"
    x = cum[kind][k]
    med = np.median(x)
    fig = plt.figure(figsize=(14, 4.5))
    hp.mollview(
        to_fullsky(x),
        fig=fig.number,
        sub=(1, 2, 1),
        min=0,
        max=vmax_abs,
        unit="visits",
        title=f"month {n}: {WHEN[kind].format(date=epoch_end_label(n))} (median = {med:.0f})",
    )
    hp.mollview(
        to_fullsky(x / max(med, 1.0)),
        fig=fig.number,
        sub=(1, 2, 2),
        min=REL_MIN,
        max=REL_MAX,
        cmap=CMAP_DIV,
        unit="N / median(N)",
        title="normalised to the median",
    )
    fig.suptitle(f"{runName} {label}")
    save_fig(fig, f"nvisits_n_time_month_{n:02d}")
    plt.show()

### Figures 1 ... N_MONTHS - Visits taken before the end of month N (one figure per month)

Files `nvisits_n_time_month_01` ... `nvisits_n_time_month_<N_MONTHS>` in `FIG_DIR`. Month 1 starts at `START_MJD`, the first day of the DP2 / pre-LSST visits.

In [ ]:
vmax_abs = max(np.percentile(cum["n_time"][-1], 99.5), 1.0)
for k in range(N_MONTHS):
    plot_month(k, vmax_abs)

## 7. Mosaics of the normalised maps

All the months on one figure (4 panels per row) for a quick look at how the non-uniformity evolves: number of visits divided by its median (`REL_MIN` - `REL_MAX`), and coadded depth minus its median (+/- `DM5_RANGE` mag). Panels have no colour bar; the colour range is given in the figure title.

In [ ]:
N_COLS = 4
N_ROWS = int(np.ceil(N_MONTHS / N_COLS))


def plot_mosaic(kind, normalise, vmin, vmax, title, fname):
    fig = plt.figure(figsize=(20, 4.5 * N_ROWS))
    for k in range(N_MONTHS):
        x = cum[kind][k]
        if normalise:
            v = x / max(np.median(x), 1.0)
        else:
            v = x - np.nanmedian(x) if np.isfinite(x).any() else x
        hp.mollview(
            to_fullsky(v),
            fig=fig.number,
            sub=(N_ROWS, N_COLS, k + 1),
            min=vmin,
            max=vmax,
            cmap=CMAP_DIV,
            cbar=False,
            title=f"month {k + 1} (until {epoch_end_label(k + 1)})",
        )
    fig.suptitle(f"{runName} {label} - {title}", fontsize=20)
    save_fig(fig, fname)
    plt.show()


plot_mosaic(
    "n_time",
    True,
    REL_MIN,
    REL_MAX,
    f"N visits / median, colour range {REL_MIN} - {REL_MAX}",
    "mosaic_n_time",
)

plot_mosaic(
    "m5_time",
    False,
    -DM5_RANGE,
    DM5_RANGE,
    f"{DEPTH_FILTER}-band coadded depth - median (mag), colour range +/-{DM5_RANGE}",
    f"mosaic_m5_time_{DEPTH_FILTER}",
)

### Figure - Mosaic of the absolute number of visits, months 1 ... N_MONTHS

Same layout for the absolute cumulative number of visits, with the same colour scale for all months (0 to the 99.5th percentile of the last month), so that the accumulation is visible. The median over the WFD pixels is given in each panel title. File `mosaic_n_time_abs`.

In [ ]:
def plot_mosaic_abs(fname):
    kind = "n_time"
    vmax = max(np.percentile(cum[kind][-1], 99.5), 1.0)
    fig = plt.figure(figsize=(20, 4.5 * N_ROWS))
    for k in range(N_MONTHS):
        x = cum[kind][k]
        hp.mollview(
            to_fullsky(x),
            fig=fig.number,
            sub=(N_ROWS, N_COLS, k + 1),
            min=0,
            max=vmax,
            cbar=False,
            title=f"month {k + 1} (median = {np.median(x):.0f})",
        )
    fig.suptitle(f"{runName} {label} - N visits, colour range 0 - {vmax:.0f}", fontsize=14)
    save_fig(fig, fname)
    plt.show()


plot_mosaic_abs("mosaic_n_time_abs")

## 8. Evolution of the WFD number of visits with the number of months

### Figure - mean, median, RMS and RMS/sqrt(mean) of N_visits versus month (`evolution_nvisits_vs_epoch`)

For each month N, statistics of the cumulative number of visits over all WFD pixels (pixels without visit counted as 0): `mean`, `median`, `RMS` (standard deviation over pixels) and `RMS / sqrt(mean)`. The last quantity is 1 for a Poisson-like scatter; values above 1 mean that the pixel-to-pixel scatter is larger than a random sampling of the same mean, i.e. the survey is less uniform than a random one. The table is saved in `DATA_DIR`.

In [ ]:
months = np.arange(1, N_MONTHS + 1)
x = cum["n_time"]
mean = x.mean(axis=1)
median = np.median(x, axis=1)
rms = x.std(axis=1)
ratio = np.where(mean > 0, rms / np.sqrt(np.maximum(mean, 1e-12)), np.nan)

fig, axs = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for ax, y, ttl in zip(
    axs.ravel(),
    [mean, median, rms, ratio],
    ["mean(N_visits)", "median(N_visits)", "RMS(N_visits)", "RMS(N_visits) / sqrt(mean(N_visits))"],
):
    ax.plot(months, y, "o-")
    ax.set_title(ttl)
    ax.grid(True, alpha=0.3)
for ax in axs[1]:
    ax.set_xlabel("epoch N (months since the first visit)")
fig.suptitle(f"{runName} {label}")
fig.tight_layout()
save_fig(fig, "evolution_nvisits_vs_epoch")
plt.show()

df_evol = pd.DataFrame(
    {
        "n": months,
        "until": [epoch_end_label(n) for n in months],
        "mean": mean,
        "median": median,
        "rms": rms,
        "rms/sqrt(mean)": ratio,
    }
)
df_evol.to_csv(os.path.join(DATA_DIR, f"evolution_nvisits_{runName}.csv"), index=False)
display(df_evol.round(3))

## 9. Uniformity versus the number of months

### Figure - Uniformity indicators for the number of visits (`uniformity_vs_epoch_nvisits`)

Four indicators from the table of section 5 as a function of the month N. Indicators that decrease with N mean that the survey becomes more uniform as the visits accumulate. Months with a median of 0 have no value (NaN) and are not drawn.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
panels = [
    ("rms/mean", "RMS / mean"),
    ("robust_sigma/median", "robust sigma / median"),
    ("(p95-p5)/median", "(p95 - p5) / median"),
    ("frac_within_10pc", "fraction of pixels within +/-10% of median"),
]
for ax, (col, ttl) in zip(axs.ravel(), panels):
    ax.plot(df_counts["n"], df_counts[col], "o-")
    ax.set_title(ttl)
    ax.grid(True, alpha=0.3)
for ax in axs[1]:
    ax.set_xlabel("epoch N (months since the first visit)")
fig.suptitle(f"{runName} {label} - number of visits")
fig.tight_layout()
save_fig(fig, "uniformity_vs_epoch_nvisits")
plt.show()

### Figure - Uniformity indicators for the coadded depth (`uniformity_vs_epoch_depth`)

Standard deviation, robust sigma and 5-95 percentile spread of the coadded depth in the `DEPTH_FILTER` band (mag), and fraction of pixels within 0.05 mag of the median, versus the month N.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
panels = [
    ("std", "std of m5 (mag)"),
    ("robust_sigma", "robust sigma of m5 (mag)"),
    ("p95-p5", "p95 - p5 of m5 (mag)"),
    ("frac_within_0.05mag", "fraction of pixels within 0.05 mag of median"),
]
for ax, (col, ttl) in zip(axs.ravel(), panels):
    ax.plot(df_depth["n"], df_depth[col], "o-")
    ax.set_title(ttl)
    ax.grid(True, alpha=0.3)
for ax in axs[1]:
    ax.set_xlabel("epoch N (months since the first visit)")
fig.suptitle(f"{runName} {label} - {DEPTH_FILTER}-band coadded depth")
fig.tight_layout()
save_fig(fig, f"uniformity_vs_epoch_depth_{DEPTH_FILTER}")
plt.show()

## 10. Distribution of the normalised number of visits

### Figure - Histograms of N / median(N) for a few months (`hist_nvisits_normalised`)

A uniform survey gives a narrow peak at 1; a survey still being built up gives a broad or multi-peaked distribution that narrows as the visits accumulate. The months shown are 1, 3, 6, 9, 12 and the last one (those larger than `N_MONTHS` are skipped).

In [ ]:
epochs_to_show = sorted({n for n in [1, 3, 6, 9, 12, N_MONTHS] if n <= N_MONTHS})
bins = np.linspace(0, 2, 81)
fig, ax = plt.subplots(figsize=(8, 5))
for n in epochs_to_show:
    x = cum["n_time"][n - 1]
    ax.hist(x / max(np.median(x), 1.0), bins=bins, histtype="step", lw=1.5, label=f"month {n}")
ax.set_xlabel("N visits / median")
ax.set_ylabel("number of pixels")
ax.grid(True, alpha=0.3)
ax.legend()
fig.suptitle(f"{runName} {label}")
fig.tight_layout()
save_fig(fig, "hist_nvisits_normalised")
plt.show()

## Notes on reading the results

- **First months**: the survey has just started and part of the WFD footprint can have no visit yet (`frac_zero`). When the median is 0 the normalised maps and most indicators are not defined (the median is replaced by 1 in the maps); this mostly shows the start-up of the survey, not a rolling cadence.
- **Season phase**: the epochs are counted in calendar time. A patch whose observing season starts just before the end of month N has a different number of visits from a patch whose season ends just after it; over about 15 months this RA-dependent term is large and is not related to the cadence strategy. The 10-year version separated it with a season-based definition, which cannot be used here.
- The cumulative **depth** reacts more slowly than the number of visits (depth scales as 1.25 log10 of the number of visits), so the same relative non-uniformity in N gives a smaller non-uniformity in magnitudes.
- Thresholds used for the 'uniform' fractions (10% in N, 0.05 mag in m5) are arbitrary; change them in `stats_counts` / `stats_depth` to match the requirement of the cosmological probe under study.